# Lecture: Integrating GNNs and LLMs

## Notebook : Preparing the Cora dataset

### Xavier Bresson


## Overview

This notebook is for students who are interested in the data pipeline construction.

Its final output is the single dataset file used by the main lab:

```text
dataset/cora/cora_prepared.pt
```

Starting from the original TAPE Cora archive, we will:

1. download the archive to a temporary directory,
2. read `cora.content`, `cora.cites`, and the title/abstract extraction files,
3. keep only the 2,708 papers that belong to the Cora graph,
4. align text, node features, labels, and graph nodes by paper ID,
5. construct an undirected citation graph with self-loops,
6. row-normalize the standard bag-of-words features,
7. create one reproducible train/validation/test split,
8. export one compact dataset-ready PyTorch file,
9. reload and verify the exported file.

The large raw extraction tree is never permanently unpacked.


## Setup

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gdown") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown>=5.2.0"])
else:
    print("gdown already installed")
    

gdown already installed


### Imports and seed

We use a fixed seed so the exported train/validation/test split is reproducible.

In [2]:
import io
import random
import tempfile
import zipfile
from pathlib import Path

import dgl
import gdown
import numpy as np
import pandas as pd
import torch

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TAPE_CORA_FILE_ID = "1hxE0OPR7VLEHesr48WisynuoNMhXJbpl"

DATA_DIR = Path("./dataset")
COMPACT_DIR = DATA_DIR / "cora"
CONTENT_PATH = COMPACT_DIR / "cora.content"
CITES_PATH = COMPACT_DIR / "cora.cites"
TEXT_PATH = COMPACT_DIR / "cora_text.csv"
PREPARED_DATA_PATH = COMPACT_DIR / "cora_prepared.pt"

DATA_DIR.mkdir(parents=True, exist_ok=True)
COMPACT_DIR.mkdir(parents=True, exist_ok=True)

print("Prepared dataset will be written to:", PREPARED_DATA_PATH)


Prepared dataset will be written to: dataset/cora/cora_prepared.pt


## Read the raw archive without extracting tens of thousands of files

TAPE's archive contains the Cora graph files together with a much larger raw-text tree. We only need:

- `cora.content`,
- `cora.cites`,
- the paper-ID-to-filename mapping,
- the extraction file for each of the 2,708 Cora papers.

The helper below finds the relevant ZIP members and extracts each paper's title and abstract in memory.


In [3]:
# Return clean title and abstract strings from one TAPE extraction file
def parse_title_abstract(raw_text):
    title = ""
    abstract = ""
    for line in raw_text.splitlines():
        line = line.strip()
        if "Title:" in line:
            title = line.split("Title:", 1)[1].strip()
        elif "Abstract:" in line:
            abstract = line.split("Abstract:", 1)[1].strip()
    return title, abstract

# Find one member by suffix inside the TAPE ZIP
def find_zip_member(names, suffix):
    matches = [name for name in names if name.endswith(suffix)]
    if not matches:
        raise RuntimeError(f"Could not find '{suffix}' inside the TAPE archive.")
    return min(matches, key=len)

# Create the three compact Cora files directly from the TAPE ZIP
def build_compact_files_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        name_set = set(names)
        content_member = find_zip_member(names, "cora.content")
        cites_member = find_zip_member(names, "cora.cites")
        papers_member = find_zip_member(names, "mccallum/cora/papers")
        root_prefix = papers_member[:-len("mccallum/cora/papers")]
        extraction_prefix = root_prefix + "mccallum/cora/extractions/"
        content_bytes = zf.read(content_member)
        cites_bytes = zf.read(cites_member)
        papers_text = zf.read(papers_member).decode("latin-1", errors="ignore")
        CONTENT_PATH.write_bytes(content_bytes)
        CITES_PATH.write_bytes(cites_bytes)
        content_text = content_bytes.decode("latin-1", errors="ignore")
        content = np.genfromtxt(io.StringIO(content_text), dtype=str)
        paper_ids = content[:, 0]
        pid_to_filename = {}
        for line in papers_text.splitlines():
            parts = line.split("\t")
            if len(parts) >= 2:
                pid_to_filename[parts[0]] = parts[1]
        rows = []
        for pid in paper_ids:
            if pid not in pid_to_filename:
                raise KeyError(f"Paper ID {pid} is missing from the papers mapping.")
            filename = pid_to_filename[pid].lstrip("/")
            member = extraction_prefix + filename
            if member not in name_set:
                suffix = "mccallum/cora/extractions/" + filename
                member = find_zip_member(names, suffix)
            raw_text = zf.read(member).decode("latin-1", errors="ignore")
            title, abstract = parse_title_abstract(raw_text)
            rows.append({"paper_id": pid, "title": title, "abstract": abstract})
        pd.DataFrame(rows).to_csv(TEXT_PATH, index=False)
        

## Download TAPE Cora and create the compact three-file representation

The ZIP is downloaded into a temporary directory. It disappears automatically after this cell finishes.

The compact local representation is
```text
dataset/cora/
├── cora.content
├── cora.cites
└── cora_text.csv
```

In [4]:
print("Downloading the official TAPE Cora archive...")

with tempfile.TemporaryDirectory(prefix="tape_cora_") as tmpdir:
    tmp_zip = Path(tmpdir) / "cora_orig.zip"
    result = gdown.download(id=TAPE_CORA_FILE_ID, output=str(tmp_zip), quiet=False)
    if result is None or not tmp_zip.exists():
        raise RuntimeError("The TAPE Cora download did not complete.")
    build_compact_files_from_zip(tmp_zip)

print("Compact files created:")
print(" ", CONTENT_PATH)
print(" ", CITES_PATH)
print(" ", TEXT_PATH)


Downloading...
From (original): https://drive.google.com/uc?id=1hxE0OPR7VLEHesr48WisynuoNMhXJbpl
From (redirected): https://drive.google.com/uc?id=1hxE0OPR7VLEHesr48WisynuoNMhXJbpl&confirm=t&uuid=19b03781-f032-412f-8956-d4aadf81603c
To: /var/folders/44/qs4c3c1d6ms37t9j12bv14240000gn/T/tape_cora__10fawwl/cora_orig.zip
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 783M/783M [02:23<00:00, 5.47MB/s]


Compact files created:
  dataset/cora/cora.content
  dataset/cora/cora.cites
  dataset/cora/cora_text.csv


## Verify the compact dataset

Cora contains 2,708 graph nodes. Each row of `cora.content` contains

$$
1\ \text{paper ID}
+1433\ \text{bag-of-words features}
+1\ \text{class label}
=1435\ \text{columns}.
$$

We also verify that the 2,708 text records have exactly the same paper IDs as `cora.content`.

In [5]:
content = np.genfromtxt(CONTENT_PATH, dtype=str)
text_df = pd.read_csv(TEXT_PATH, dtype={"paper_id": str}, keep_default_na=False)

assert content.shape == (2708, 1435), content.shape
assert len(text_df) == 2708, len(text_df)
assert text_df["paper_id"].nunique() == 2708
assert set(text_df["paper_id"]) == set(content[:, 0])

print("Compact dataset verified.")
print("cora.content shape:", content.shape)
print("text records:", len(text_df))


Compact dataset verified.
cora.content shape: (2708, 1435)
text records: 2708


## Build graph features, labels and the reproducible split

For node $i$, the standard Cora bag-of-words feature vector is row-normalized as

$$
\tilde{\mathbf{x}}_i
=
\frac{\mathbf{x}_i}
{\max\!\left(1,\sum_j x_{ij}\right)}.
$$

Every citation is inserted in both directions, duplicate edges are removed, and every node receives a self-loop.

For $n$ nodes, the split is

$$
n_{\text{train}}=\lfloor 0.10n\rfloor,
\qquad
n_{\text{val}}=\lfloor 0.40n\rfloor,
\qquad
n_{\text{test}}=n-n_{\text{train}}-n_{\text{val}}.
$$

In [6]:
# Load the Cora paper IDs and convert the bag-of-words features into a PyTorch tensor
content = np.genfromtxt(CONTENT_PATH, dtype=str)
paper_ids = content[:, 0]
feature_values = content[:, 1:-1].astype(np.float32)
x_standard = torch.tensor(feature_values, dtype=torch.float32)

class_names = [
    "Case_Based",
    "Genetic_Algorithms",
    "Neural_Networks",
    "Probabilistic_Methods",
    "Reinforcement_Learning",
    "Rule_Learning",
    "Theory",
]

# Map the seven class names to integer labels and convert them into a target tensor
class_to_idx = {name: i for i, name in enumerate(class_names)}
label_indices = [class_to_idx[label] for label in content[:, -1]]
y = torch.tensor(label_indices, dtype=torch.long)

# Map paper IDs to node indices and convert citations into bidirectional graph edges
id_to_idx = {paper_id: i for i, paper_id in enumerate(paper_ids)}
raw_edges = np.genfromtxt(CITES_PATH, dtype=str)
src = []
dst = []
for a, b in raw_edges:
    if a in id_to_idx and b in id_to_idx:
        ia = id_to_idx[a]
        ib = id_to_idx[b]
        src.extend([ia, ib])
        dst.extend([ib, ia])

# Build a simplified DGL graph, remove existing self-loops, and add one self-loop per node
src_tensor = torch.tensor(src, dtype=torch.int64)
dst_tensor = torch.tensor(dst, dtype=torch.int64)
g = dgl.graph((src_tensor, dst_tensor), num_nodes=len(paper_ids))
g = dgl.to_simple(g)
g = dgl.remove_self_loop(g)
g = dgl.add_self_loop(g)

# Row-normalize each node's bag-of-words feature vector so its entries sum to one
row_sum = x_standard.sum(dim=1, keepdim=True).clamp(min=1.0)
x_standard = x_standard / row_sum

# Create a reproducible random 10%/40%/50% train/validation/test split using boolean masks
num_nodes = g.num_nodes()
generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(num_nodes, generator=generator)
n_train = int(0.10 * num_nodes)
n_val = int(0.40 * num_nodes)
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

print(g)
print("standard features:", tuple(x_standard.shape))
print("train/val/test:", train_mask.sum().item(), val_mask.sum().item(), test_mask.sum().item())


Graph(num_nodes=2708, num_edges=13264,
      ndata_schemes={}
      edata_schemes={'count': Scheme(shape=(), dtype=torch.int64)})
standard features: (2708, 1433)
train/val/test: 270 1083 1355


## Align each graph node with its title and abstract

The alignment key is the original Cora `paper_id`. For every node $i$ we require

$$
\text{text}_i
\longleftrightarrow
\mathbf{x}^{\text{BoW}}_i
\longleftrightarrow
y_i
\longleftrightarrow
\text{graph node }i.
$$

This prevents a text embedding from being attached to the wrong paper.

In [7]:
# Load the paper text data and align each title and abstract with the graph-node order
text_df = pd.read_csv(TEXT_PATH, dtype={"paper_id": str}, keep_default_na=False)
text_by_id = text_df.set_index("paper_id")
titles = []
abstracts = []
for pid in paper_ids:
    titles.append(text_by_id.loc[pid, "title"])
    abstracts.append(text_by_id.loc[pid, "abstract"])

# Combine each paper's title and abstract into the text string used by the language model
texts = []
for title, abstract in zip(titles, abstracts):
    texts.append(f"Title: {title}\nAbstract: {abstract}")

assert len(texts) == num_nodes == 2708
assert x_standard.shape == (2708, 1433)

print("Number of aligned texts:", len(texts))
print("\nExample node 0:\n")
print(texts[0][:800])


Number of aligned texts: 2708

Example node 0:

Title: The megaprior heuristic for discovering protein sequence patterns
Abstract: Several computer algorithms for discovering patterns in groups of protein sequences are in use that are based on fitting the parameters of a statistical model to a group of related sequences. These include hidden Markov model (HMM) algorithms for multiple sequence alignment, and the MEME and Gibbs sampler algorithms for discovering motifs. These algorithms are sometimes prone to producing models that are incorrect because two or more patterns have been combined. The statistical model produced in this situation is a convex combination (weighted average) of two or more different models. This paper presents a solution to the problem of convex combinations in the form of a heuristic based on using extremely low 


## Export one dataset-ready file for the main lab

We prepare the final graph edges, features, labels, masks, paper IDs, class names, and aligned text into one PyTorch file.


In [8]:
prepared_src, prepared_dst = g.edges()

prepared = {}
prepared["version"] = 1
prepared["num_nodes"] = num_nodes
prepared["src"] = prepared_src.cpu()
prepared["dst"] = prepared_dst.cpu()
prepared["x_standard"] = x_standard.cpu()
prepared["y"] = y.cpu()
prepared["train_mask"] = train_mask.cpu()
prepared["val_mask"] = val_mask.cpu()
prepared["test_mask"] = test_mask.cpu()
prepared["texts"] = texts
prepared["paper_ids"] = paper_ids.tolist()
prepared["class_names"] = class_names

torch.save(prepared, PREPARED_DATA_PATH)

size_mb = PREPARED_DATA_PATH.stat().st_size / 1024**2
print("Saved:", PREPARED_DATA_PATH)
print(f"Size: {size_mb:.2f} MB")


Saved: dataset/cora/cora_prepared.pt
Size: 17.42 MB
